# Campaign Sales Ingestion (Simulated)

Reads the **simulated** `campaign_sales_simulated.csv` table -- impressions,
clicks, conversions, and revenue invented by
`fabric/sample-data/generate_synthetic_sales.py` -- from a OneLake shortcut
and loads it as a Delta table, then joins it with the real
`campaign_insights` table (populated by `campaign_insights_ingestion.ipynb`
from actual Content Understanding analyzer output) for a combined
"analyzer attributes + simulated performance" view in Power BI.

**Every row in `campaign_sales_simulated` is synthetic demo data.** No real
Progressive sales, media spend, or performance figures are used anywhere in
this repository. See the docstring in `generate_synthetic_sales.py` and
`fabric/README.md` for the simulation methodology.

No credentials are configured in this notebook; the shortcut authenticates
with the Fabric workspace identity (Entra ID, no SAS/keys), same as
`campaign_insights_ingestion.ipynb`.

Run order:
1. Run `campaign_insights_ingestion.ipynb` at least once (creates `campaign_insights`).
2. Create a shortcut (or upload directly into Files) named e.g. `Files/campaign-sales`
   pointing at the `campaign_sales_simulated.csv` file -- see fabric/README.md.
3. Run this notebook to (re)build `campaign_sales_simulated` and the joined
   `campaign_performance_simulated` table.


In [ ]:
# Path to the file (or shortcut folder) holding campaign_sales_simulated.csv.
# Update SALES_CSV_PATH if you name the shortcut/Lakehouse path differently.
SALES_CSV_PATH = "Files/campaign-sales/campaign_sales_simulated.csv"
SALES_DELTA_TABLE_NAME = "campaign_sales_simulated"
JOINED_DELTA_TABLE_NAME = "campaign_performance_simulated"
INSIGHTS_DELTA_TABLE_NAME = "campaign_insights"


## Load the simulated sales CSV into a Delta table

This is a small, fully-regenerated reference table (not an append-only
stream like `campaign_insights`), so each run simply overwrites it from the
current CSV.


In [ ]:
sales_df = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(SALES_CSV_PATH)
)

print(f"Loaded {sales_df.count()} simulated campaign sales row(s) from {SALES_CSV_PATH}")
sales_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    SALES_DELTA_TABLE_NAME
)


## Join with `campaign_insights` on `campaign_name` + result id

`campaign_insights.file_path` carries the result `id` as its filename (e.g.
`cu-results/<campaign_name>/<id>.json`); extract it so rows join cleanly to
`campaign_sales_simulated.result_id` from the same synthetic video.


In [ ]:
from pyspark.sql.functions import regexp_extract, col

insights_df = spark.read.table(INSIGHTS_DELTA_TABLE_NAME).withColumn(
    "result_id", regexp_extract(col("file_path"), r"([0-9a-fA-F-]+)\.json$", 1)
)

joined_df = insights_df.join(
    sales_df, on=["campaign_name", "result_id"], how="inner"
)

print(
    f"Joined {joined_df.count()} row(s) across "
    f"{insights_df.count()} campaign_insights row(s) and {sales_df.count()} simulated sales row(s)."
)
joined_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    JOINED_DELTA_TABLE_NAME
)


## Sanity-check the simulated correlation

Quick aggregate to confirm the simulation behaves as documented: higher
click-through rate for `Positive` sentiment than `Mixed`, and higher
conversion rate for a direct `GetAQuote`/`PhoneCall` ask. This is diagnostic
output for building the demo narrative, not a Power BI table.


In [ ]:
joined_df.groupBy("sentiment_category").avg("click_through_rate").orderBy(
    "sentiment_category"
).show()

joined_df.groupBy("call_to_action_type").avg("conversion_rate").orderBy(
    "call_to_action_type"
).show()


## Next steps

- Build a Power BI report/semantic model on `campaign_performance_simulated`:
  revenue and conversion rate by `campaign_name`, `sentiment_category`, and
  `call_to_action_type`.
- If you generate the v2-schema results (`generate_synthetic_commercials_v2.py`
  / `cuCommercialVideoAnalyzerV2`), ingest `cu-results-v2` the same way as
  `campaign_insights_ingestion.ipynb` (same flattener, different shortcut
  path/table name) and compare the v1 free-text `CallToAction`/
  `EmotionSentiment` columns against the v2 classified
  `CallToActionType`/`SentimentCategory` columns for the *same* `result_id`,
  to demo how a schema change changes what's directly chartable without
  text parsing.
- Keep re-running `generate_synthetic_sales.py` with a new `--seed` to
  produce a different (still deterministic per-seed) simulated scenario for
  the demo.
